In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
#将数据输出结果导出
import os

In [ ]:
df=pd.read_csv('data/clean_2019-Nov.csv')
'''-------------转化漏斗多维度拆解计算整体漏斗转化率:计算各环节独立用户数、单步转化率、总转化率、流失率-------------'''

user_act = df.groupby('user_id')['event_type'].agg(
    has_pv=lambda x: 'view' in x.values,
    has_cart=lambda x: 'cart' in x.values,
    has_buy=lambda x: 'purchase' in x.values
).reset_index()

# 各环节人数
pv_cnt = user_act['has_pv'].sum()
cart_cnt = user_act['has_cart'].sum()
buy_cnt = user_act['has_buy'].sum()

# 转化率
# 各环节独立用户数
pv_cnt = df[df['event_type'] == 'view']['user_id'].nunique()
cart_cnt = df[df['event_type'] == 'cart']['user_id'].nunique()
buy_cnt = df[df['event_type'] == 'purchase']['user_id'].nunique()
# 购买用户：同时有加购+购买行为的用户
buy_user_ids = df[df['event_type'] == 'purchase']['user_id'].unique()
cart_user_ids = df[df['event_type'] == 'cart']['user_id'].unique()
buy_cnt = len(set(buy_user_ids) & set(cart_user_ids))

# 转化率计算
pv_to_cart = np.divide(cart_cnt, pv_cnt, where=pv_cnt!=0, out=np.zeros_like(cart_cnt, dtype=float))
cart_to_buy = np.divide(buy_cnt, cart_cnt, where=cart_cnt!=0, out=np.zeros_like(cart_cnt, dtype=float))
pv_to_buy = np.divide(buy_cnt, pv_cnt, where=pv_cnt!=0, out=np.zeros_like(buy_cnt, dtype=float))

# 2. 画漏斗图
stages = ['浏览', '加购', '购买']
values = [pv_cnt, cart_cnt, buy_cnt]

plt.figure(figsize=(8, 4))

# 从高到低横向排列
plt.barh(stages[::-1], values[::-1], color='gray')

for i, v in enumerate(values[::-1]):
    plt.text(
        v + max(values) * 0.02,
        i,
        str(v),
        va='center'
    )

plt.title('用户转化漏斗')
plt.xlabel('用户数')
plt.ylabel('转化环节')
plt.xlim(0, max(values) * 1.15)
plt.tight_layout()

plt.show()
print("="*50)
print(" 用户转化漏斗结果:")
print(f"浏览用户数：{pv_cnt} 人")
print(f"加购用户数：{cart_cnt} 人")
print(f"购买用户数：{buy_cnt} 人")
print("-"*30)
print(f"浏览→加购转化率：{pv_to_cart:.4f}（{pv_to_cart*100:.2f}%）")
print(f"加购→购买转化率：{cart_to_buy:.4f}（{cart_to_buy*100:.2f}%）")
print(f"浏览→购买整体转化率：{pv_to_buy:.4f}（{pv_to_buy*100:.2f}%）")
# ====================== 保存数据用于后续可视化输出 ======================
# 拼接漏斗数据表
df_funnel = pd.DataFrame({
    'stage': ['浏览', '加购', '购买'],
    'user_count': [pv_cnt, cart_cnt, buy_cnt]
})
df_funnel.to_csv('output/data/conversion_funnel.csv',
                 index=False, encoding='utf-8-sig')
